<a href="https://colab.research.google.com/github/luigidaniel1113-cpu/Sistemas-Operativo/blob/main/Echeverria%20Delgado%20Luis%20Daniel_Pratctica_Sistemas_Operativos_U2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 💻 Sistemas Operativos - Administración de Procesos - Jearaquía, Implantación, IPC
**Asignatura:** Fundamentos y Diseño de Sistemas Operativos (6.º Semestre)    
**Unidad II:** Administración de Procesos  
**Tiempo estimado:** 2 horas (Asíncrono)  
**Repositorio Base:** Integración con GitHub  

---

## 1. Objetivos de Aprendizaje
1. Examinar la tabla de procesos del sistema operativo huésped utilizando comandos nativos de Linux (`ps`) en Google Colab para identificar atributos clave como identificadores de procesos (**PID**/**PPID**) y relaciones jerárquicas.

2. Explorar la estructura de datos interna que el kernel de Linux expone a través del sistema de archivos virtual (`/proc`) para comprender cómo se recopila y expone la información de ejecución a nivel de sistema.

3. Analizar la jerarquía de procesos y las relaciones de descendencia en un entorno Linux real empleando herramientas como pstree y el sistema de archivos virtual `/proc` en Google Colab.

4. Implementar las llamadas al sistema clásicas de control de procesos (`fork`, `exec`, `exit`, `wait`) mediante un simulador orientado a objetos en C++17.

5. Examinar fenómenos fundamentales de los sistemas operativos, tales como la persistencia de metadatos en estados Zombie y la adopción automática de procesos huérfanos por parte del proceso raíz (`init`).

6. Examinar los recursos de IPC activos en el sistema operativo huésped (**memoria compartida, semáforos y colas de mensajes**) mediante el comando `ipcs` y parámetros del kernel en `/proc`.

7. Implementar el control de concurrencia y la exclusión mutua aplicando **semáforos** ***POSIX*** (`sem_t`, `sem_wait`, `sem_post`) para resolver el problema clásico del **Productor-Consumidor** con un buffer circular.

8. Desarrollar canales de comunicación unidireccional entre procesos independientes utilizando las llamadas al sistema `pipe()`, `fork()`, `read()` y `write()`.

---

## 2. Sección Práctica 1: Inspección del Sistema Operativo.

En esta sección analizarás los fundamentos de la administración de procesos en los sistemas operativos modernos mediante la inspección de la tabla de procesos en un entorno Linux real(Servidor Linux provisto por Google).

### [CÓDIGO 2.1] inspeccionar la tabla de procesos real del sistema operativo huésped
```bash
# 1. Visualizar la lista completa de procesos activos en formato detallado
!ps -aux

# 2. Inspeccionar la jerarquía de procesos y ver el proceso PID 1 (init/systemd)
!ps -ef --forest

# 3. Consultar la estructura de datos interna que Linux expone en el sistema de archivos /proc para un PID
!ls -l /proc/1/
!cat /proc/1/status | head -n 15
```

In [ ]:
# 1. Visualizar la lista completa de procesos activos en formato detallado
!ps -aux

# 2. Inspeccionar la jerarquía de procesos y ver el proceso PID 1 (init/systemd)
!ps -ef --forest

# 3. Consultar la estructura de datos interna que Linux expone en el sistema de archivos /proc para un PID
!ls -l /proc/1/
!cat /proc/1/status | head -n 15

USER         PID %CPU %MEM    VSZ   RSS TTY      STAT START   TIME COMMAND
root           1  0.0  0.0    984   560 ?        Ss   18:20   0:00 /sbin/docker-
root           7  0.4  0.4 1161180 57588 ?       Sl   18:20   0:01 /tools/node/b
root           9  0.0  0.0   7344  3512 ?        S    18:20   0:00 /bin/bash -e 
root          11  0.0  0.0   7344  2088 ?        S    18:20   0:00 /bin/bash -e 
root          12  0.0  0.2 1286396 33396 ?       Sl   18:20   0:00 /usr/colab/bi
root          40  0.0  0.0   5720  2280 ?        Ss   18:20   0:00 tail -n +0 -F
root          41  0.0  0.0   5720  2272 ?        Ss   18:20   0:00 tail -n +0 -F
root          65  5.8  0.0      0     0 ?        Z    18:20   0:19 [python3] <de
root          66  0.8  0.5 101772 79228 ?        S    18:20   0:02 python3 /usr/
root          87  0.0  0.0  12024  2848 ?        Ss   18:20   0:00 sshd: /usr/sb
root         102  2.3  1.2 556876 163752 ?       Sl   18:21   0:07 /usr/bin/pyth
root        1415 73.1  0.7 654628 

### [CÓDIGO 2.2] Implementación de la Estructura struct BCP en C++17
```cpp
%%writefile bcp_struct.cpp
// ==============================================================================
// EJERCICIO PRÁCTICO: ESTRUCTURA DEL BLOQUE DE CONTROL DE PROCESO (BCP)
// Lenguaje: C++17
// Asignatura: Sistemas Operativos - Unidad II: Administración de Procesos
// ==============================================================================

#include <iostream>
#include <string>
#include <vector>
#include <iomanip>
#include <cstdint>

// Enumeración de los Estados Conceptuales de un Proceso (Modelo de 5 Estados)
enum class EstadoProceso {
    NUEVO,
    LISTO,
    EJECUTANDO,
    BLOQUEADO,
    SALIENTE
};

// Función auxiliar para convertir el Enum a String legible
std::string estadoToString(EstadoProceso estado) {
    switch (estado) {
        case EstadoProceso::NUEVO:      return "NUEVO";
        case EstadoProceso::LISTO:      return "LISTO";
        case EstadoProceso::EJECUTANDO: return "EJECUTANDO";
        case EstadoProceso::BLOQUEADO:  return "BLOQUEADO";
        case EstadoProceso::SALIENTE:   return "SALIENTE";
        default:                        return "DESCONOCIDO";
    }
}

// Estructura de Datos del BCP (Process Control Block)
struct BCP {
    // 1. Identificadores
    uint32_t pid;                   // Process ID único
    uint32_t ppid;                  // Parent Process ID
    std::string nombre_programa;    // Nombre del ejecutable en disco

    // 2. Información de Estado del Procesador (Contexto Guardado)
    EstadoProceso estado_actual;    // Estado conceptual del proceso
    uint16_t PC;                    // Program Counter (Contador de Programa)
    uint16_t SP;                    // Stack Pointer (Puntero de Pila)
    int AC;                         // Acumulador
    uint8_t PSW_modo;               // 0 = Modo Usuario, 1 = Modo Kernel

    // 3. Información de Gestión de Memoria y Planificación
    uint32_t direccion_base_ram;    // Dirección inicial de memoria asignada
    uint32_t limite_memoria_bytes;  // Tamaño del espacio de direcciones
    int prioridad;                  // Nivel de prioridad de planificación

    // Método para imprimir la ficha técnica del BCP
    void mostrarBCP() const {
        std::cout << "┌" << std::string(65, '-') << "┐\n";
        std::cout << "│ FICHA TÉCNICA DEL BCP (PID: " << std::setw(5) << pid << ") - Program: "
                  << std::left << std::setw(20) << nombre_programa << "│\n";
        std::cout << "├" << std::string(65, '-') << "┤\n";
        std::cout << "│ Estado Actual  : " << std::setw(15) << estadoToString(estado_actual)
                  << " | PPID Padre     : " << std::setw(10) << ppid << " │\n";
        std::cout << "│ Contexto CPU   : PC=0x" << std::hex << std::uppercase << std::setw(4) << std::setfill('0') << PC << std::dec
                  << " | SP=0x" << std::hex << std::uppercase << std::setw(4) << std::setfill('0') << SP << std::dec
                  << " | AC=" << std::setw(6) << std::setfill(' ') << AC << " │\n";
        std::cout << "│ Modo PSW       : " << (PSW_modo == 1 ? "KERNEL (1)    " : "USUARIO (0)   ")
                  << " | Prioridad      : " << std::setw(10) << prioridad << " │\n";
        std::cout << "│ Memoria Asign. : Base=0x" << std::hex << std::uppercase << std::setw(6) << std::setfill('0') << direccion_base_ram << std::dec
                  << " | Límite=" << std::setw(6) << limite_memoria_bytes << " Bytes  │\n";
        std::cout << "└" << std::string(65, '-') << "┘\n";
    }
};

int main() {
    std::cout << "=================================================================\n";
    std::cout << "   SIMULACIÓN DE LA TABLA DE PROCESOS DEL SISTEMA OPERATIVO     \n";
    std::cout << "=================================================================\n\n";

    // Creación de una Tabla de Procesos simulada (Vector de BCPs)
    std::vector<BCP> tabla_procesos;

    // Inicialización del Proceso 1 (Navegador)
    BCP p1;
    p1.pid = 101;
    p1.ppid = 1; // Hijo del proceso init
    p1.nombre_programa = "chrome.exe";
    p1.estado_actual = EstadoProceso::EJECUTANDO;
    p1.PC = 0x0420;
    p1.SP = 0x0FFF;
    p1.AC = 150;
    p1.PSW_modo = 0; // Modo Usuario
    p1.direccion_base_ram = 0x10000;
    p1.limite_memoria_bytes = 65536;
    p1.prioridad = 5;

    // Inicialización del Proceso 2 (Compilador Bloqueado por E/S)
    BCP p2;
    p2.pid = 102;
    p2.ppid = 101;
    p2.nombre_programa = "g++_compiler";
    p2.estado_actual = EstadoProceso::BLOQUEADO;
    p2.PC = 0x08A2;
    p2.SP = 0x0E00;
    p2.AC = 0;
    p2.PSW_modo = 0; // Modo Usuario
    p2.direccion_base_ram = 0x20000;
    p2.limite_memoria_bytes = 131072;
    p2.prioridad = 10;

    // Agregar procesos a la tabla
    tabla_procesos.push_back(p1);
    tabla_procesos.push_back(p2);

    // Desplegar la información guardada en los BCPs
    for (const auto& bcp : tabla_procesos) {
        bcp.mostrarBCP();
        std::cout << "\n";
    }

    return 0;
}
```

In [ ]:
%%writefile bcp_struct.cpp
// ==============================================================================
// EJERCICIO PRÁCTICO: ESTRUCTURA DEL BLOQUE DE CONTROL DE PROCESO (BCP)
// Lenguaje: C++17
// Asignatura: Sistemas Operativos - Unidad II: Administración de Procesos
// ==============================================================================

# include <iostream>
# include <string>
# include <vector>
# include <iomanip>
# include <cstdint>

// Enumeración de los Estados Conceptuales de un Proceso (Modelo de 5 Estados)
enum class EstadoProceso {
    NUEVO,
    LISTO,
    EJECUTANDO,
    BLOQUEADO,
    SALIENTE
};

// Función auxiliar para convertir el Enum a String legible
std::string estadoToString(EstadoProceso estado) {
    switch (estado) {
        case EstadoProceso::NUEVO:      return "NUEVO";
        case EstadoProceso::LISTO:      return "LISTO";
        case EstadoProceso::EJECUTANDO: return "EJECUTANDO";
        case EstadoProceso::BLOQUEADO:  return "BLOQUEADO";
        case EstadoProceso::SALIENTE:   return "SALIENTE";
        default:                        return "DESCONOCIDO";
    }
}

// Estructura de Datos del BCP (Process Control Block)
struct BCP {
    // 1. Identificadores
    uint32_t pid;                   // Process ID único
    uint32_t ppid;                  // Parent Process ID
    std::string nombre_programa;    // Nombre del ejecutable en disco

    // 2. Información de Estado del Procesador (Contexto Guardado)
    EstadoProceso estado_actual;    // Estado conceptual del proceso
    uint16_t PC;                    // Program Counter (Contador de Programa)
    uint16_t SP;                    // Stack Pointer (Puntero de Pila)
    int AC;                         // Acumulador
    uint8_t PSW_modo;               // 0 = Modo Usuario, 1 = Modo Kernel

    // 3. Información de Gestión de Memoria y Planificación
    uint32_t direccion_base_ram;    // Dirección inicial de memoria asignada
    uint32_t limite_memoria_bytes;  // Tamaño del espacio de direcciones
    int prioridad;                  // Nivel de prioridad de planificación

    // Método para imprimir la ficha técnica del BCP
    void mostrarBCP() const {
        std::cout << "┌" << std::string(65, '-') << "┐\n";
        std::cout << "│ FICHA TÉCNICA DEL BCP (PID: " << std::setw(5) << pid << ") - Program: "
                  << std::left << std::setw(20) << nombre_programa << "│\n";
        std::cout << "├" << std::string(65, '-') << "┤\n";
        std::cout << "│ Estado Actual  : " << std::setw(15) << estadoToString(estado_actual)
                  << " | PPID Padre     : " << std::setw(10) << ppid << " │\n";
        std::cout << "│ Contexto CPU   : PC=0x" << std::hex << std::uppercase << std::setw(4) << std::setfill('0') << PC << std::dec
                  << " | SP=0x" << std::hex << std::uppercase << std::setw(4) << std::setfill('0') << SP << std::dec
                  << " | AC=" << std::setw(6) << std::setfill(' ') << AC << " │\n";
        std::cout << "│ Modo PSW       : " << (PSW_modo == 1 ? "KERNEL (1)    " : "USUARIO (0)   ")
                  << " | Prioridad      : " << std::setw(10) << prioridad << " │\n";
        std::cout << "│ Memoria Asign. : Base=0x" << std::hex << std::uppercase << std::setw(6) << std::setfill('0') << direccion_base_ram << std::dec
                  << " | Límite=" << std::setw(6) << limite_memoria_bytes << " Bytes  │\n";
        std::cout << "└" << std::string(65, '-') << "┘\n";
    }
};

int main() {
    std::cout << "=================================================================\n";
    std::cout << "   SIMULACIÓN DE LA TABLA DE PROCESOS DEL SISTEMA OPERATIVO     \n";
    std::cout << "=================================================================\n\n";

    // Creación de una Tabla de Procesos simulada (Vector de BCPs)
    std::vector<BCP> tabla_procesos;

    // Inicialización del Proceso 1 (Navegador)
    BCP p1;
    p1.pid = 101;
    p1.ppid = 1; // Hijo del proceso init
    p1.nombre_programa = "chrome.exe";
    p1.estado_actual = EstadoProceso::EJECUTANDO;
    p1.PC = 0x0420;
    p1.SP = 0x0FFF;
    p1.AC = 150;
    p1.PSW_modo = 0; // Modo Usuario
    p1.direccion_base_ram = 0x10000;
    p1.limite_memoria_bytes = 65536;
    p1.prioridad = 5;

    // Inicialización del Proceso 2 (Compilador Bloqueado por E/S)
    BCP p2;
    p2.pid = 102;
    p2.ppid = 101;
    p2.nombre_programa = "g++_compiler";
    p2.estado_actual = EstadoProceso::BLOQUEADO;
    p2.PC = 0x08A2;
    p2.SP = 0x0E00;
    p2.AC = 0;
    p2.PSW_modo = 0; // Modo Usuario
    p2.direccion_base_ram = 0x20000;
    p2.limite_memoria_bytes = 131072;
    p2.prioridad = 10;

    // Agregar procesos a la tabla
    tabla_procesos.push_back(p1);
    tabla_procesos.push_back(p2);

    // Desplegar la información guardada en los BCPs
    for (const auto& bcp : tabla_procesos) {
        bcp.mostrarBCP();
        std::cout << "\n";
    }

    return 0;
}

Overwriting bcp_struct.cpp


### [CÓDIGO 2.3] Compilación y ejecución del código
```bash
# Compilar el código en C++17 con el máximo nivel de optimización (-O3)
!g++ -O3 -std=c++17 bcp_struct.cpp -o bcp_struct

# Ejecutar la simulación
!./bcp_struct
```

In [ ]:
# Compilar el código en C++17 con el máximo nivel de optimización (-O3)
!g++ -O3 -std=c++17 bcp_struct.cpp -o bcp_struct

# Ejecutar la simulación
!./bcp_struct

   SIMULACIÓN DE LA TABLA DE PROCESOS DEL SISTEMA OPERATIVO     

┌-----------------------------------------------------------------┐
│ FICHA TÉCNICA DEL BCP (PID:   101) - Program: chrome.exe          │
├-----------------------------------------------------------------┤
│ Estado Actual  : EJECUTANDO      | PPID Padre     : 1          │
│ Contexto CPU   : PC=0x4200 | SP=0xFFF0 | AC=150    │
│ Modo PSW       : USUARIO (0)    | Prioridad      : 5          │
│ Memoria Asign. : Base=0x100000 | Límite=655360 Bytes  │
└-----------------------------------------------------------------┘

┌-----------------------------------------------------------------┐
│ FICHA TÉCNICA DEL BCP (PID: 10200) - Program: g++_compiler00000000│
├-----------------------------------------------------------------┤
│ Estado Actual  : BLOQUEADO000000 | PPID Padre     : 1010000000 │
│ Contexto CPU   : PC=0x8A20 | SP=0xE000 | AC=0      │
│ Modo PSW       : USUARIO (0)    | Prioridad      : 10         │
│ Memoria Asign. : 

## 3. Sección Práctica 2: Jerarquía de Procesos.  

En esta sección explorarás y comprenderás la jerarquía, el ciclo de vida y la gestión de procesos en sistemas operativos mediante la inspección de estructuras del kernel en Linux y la simulación algorítmica de llamadas al sistema (`fork`, `exec`, `exit`, `wait`)

### [CÓDIGO 3.1] Inspección de la Jerarquía de Procesos en Linux.

Ejecutar los siguientes comandos en celdas de código bash en Google Colab:
```bash
# 1. Mostrar la jerarquía de procesos en formato de árbol (Tree)
!pstree -p

# 2. Consultar el BCP expuesto por Linux en /proc para el proceso actual
!cat /proc/self/status | grep -E "Name|State|Pid|PPid|Threads"

# 3. Identificar el proceso raíz init/systemd (PID 1)
!ps -p 1 -o pid,ppid,comm,state
```

In [ ]:
# 1. Mostrar la jerarquía de procesos en formato de árbol (Tree)
!pstree -p

# 2. Consultar el BCP expuesto por Linux en /proc para el proceso actual
!cat /proc/self/status | grep -E "Name|State|Pid|PPid|Threads"

# 3. Identificar el proceso raíz init/systemd (PID 1)
!ps -p 1 -o pid,ppid,comm,state

docker-init(1)─┬─node(7)─┬─colab-fileshim.(66)
               │         ├─jupyter-server(102)─┬─python3(1415)─┬─pstree(2945)
               │         │                     │               ├─{python3}(1423)
               │         │                     │               ├─{python3}(1424)
               │         │                     │               ├─{python3}(1425)
               │         │                     │               ├─{python3}(1426)
               │         │                     │               ├─{python3}(1427)
               │         │                     │               ├─{python3}(1428)
               │         │                     │               ├─{python3}(1429)
               │         │                     │               ├─{python3}(1430)
               │         │                     │               └─{python3}(1435)
               │         │                     ├─{jupyter-server}(120)
               │         │                     ├─{jupyter-server}(1409)
   

### [CÓDIGO 3.2] Inspección de la Jerarquía de Procesos en Linux.

Simulación de Jerarquía, BCP y Cambio de Contexto, copiar y ejecutar el siguiente código en Colab:

```cpp
%%writefile laboratorio_jerarquia_bcp.cpp
// ==============================================================================
// EJERCICIO PRÁCTICO: JERARQUÍA DE PROCESOS Y CAMBIO DE CONTEXTO
// Lenguaje: C++17
// Asignatura: Sistemas Operativos - Unidad II: Administración de Procesos
// ==============================================================================

#include <iostream>
#include <vector>
#include <string>
#include <iomanip>
#include <memory>

enum class Estado { NUEVO, LISTO, EJECUTANDO, BLOQUEADO, ZOMBIE, TERMINADO };

std::string estadoToString(Estado e) {
    switch (e) {
        case Estado::NUEVO:      return "NUEVO";
        case Estado::LISTO:      return "LISTO";
        case Estado::EJECUTANDO: return "EJECUTANDO";
        case Estado::BLOQUEADO:  return "BLOQUEADO";
        case Estado::ZOMBIE:     return "ZOMBIE";
        case Estado::TERMINADO:  return "TERMINADO";
        default:                 return "DESCONOCIDO";
    }
}

// Estructura del Bloque de Control de Proceso (BCP)
struct BCP {
    int pid;
    int ppid;
    std::string nombre_programa;
    Estado estado;
    uint16_t PC; // Program Counter
    uint16_t SP; // Stack Pointer
    int PSW_modo; // 0 = Usuario, 1 = Kernel
    int codigo_salida;

    void imprimir() const {
        std::cout << "[BCP PID=" << std::setw(3) << pid
                  << " | PPID=" << std::setw(3) << ppid
                  << " | Prog: " << std::left << std::setw(12) << nombre_programa
                  << " | Estado: " << std::setw(10) << estadoToString(estado)
                  << " | PC: 0x" << std::hex << std::uppercase << std::setw(4) << std::setfill('0') << PC << std::dec
                  << " | Modo: " << (PSW_modo == 1 ? "KERNEL" : "USUARIO") << "]\n";
    }
};

// Tabla de Procesos Simulada del Sistema Operativo
class SistemaOperativoSimulado {
private:
    std::vector<BCP> tabla_procesos;
    int contador_pid = 100;
    int pid_activo = -1;

public:
    SistemaOperativoSimulado() {
        // Crear proceso raíz (init - PID 1)
        BCP init{1, 0, "init", Estado::EJECUTANDO, 0x0010, 0x0FFF, 1, 0};
        tabla_procesos.push_back(init);
        pid_activo = 1;
    }

    // Simulación de la llamada fork()
    int simular_fork(int pid_padre, const std::string& nombre_hijo) {
        BCP* padre = obtenerBCP(pid_padre);
        if (!padre) return -1;

        contador_pid++;
        BCP hijo;
        hijo.pid = contador_pid;
        hijo.ppid = padre->pid;
        hijo.nombre_programa = nombre_hijo;
        hijo.estado = Estado::LISTO;
        hijo.PC = padre->PC; // Copia el PC del padre (misma instrucción)
        hijo.SP = padre->SP - 0x0100; // Nueva pila
        hijo.PSW_modo = 0;
        hijo.codigo_salida = 0;

        tabla_procesos.push_back(hijo);
        std::cout << "  [FORK] Proceso Padre PID " << pid_padre
                  << " creó al Proceso Hijo PID " << hijo.pid << " (" << nombre_hijo << ")\n";
        return hijo.pid;
    }

    // Simulación de la llamada exec()
    bool simular_exec(int pid, const std::string& nuevo_programa) {
        BCP* proc = obtenerBCP(pid);
        if (!proc) return false;

        std::cout << "  [EXEC] Proceso PID " << pid << " reemplaza su imagen por '" << nuevo_programa << "'\n";
        proc->nombre_programa = nuevo_programa;
        proc->PC = 0x1000; // Reinicia PC al punto de entrada del nuevo ejecutable
        return true;
    }

    // Simulación de terminación exit() y estado Zombie
    void simular_exit(int pid, int status) {
        BCP* proc = obtenerBCP(pid);
        if (!proc) return;

        proc->codigo_salida = status;
        proc->estado = Estado::ZOMBIE;
        std::cout << "  [EXIT] Proceso PID " << pid << " finalizó con código " << status
                  << " -> Entra a Estado ZOMBIE (esperando wait del padre)\n";
    }

    // Simulación de la llamada wait() recolectora
    void simular_wait(int pid_padre) {
        for (auto it = tabla_procesos.begin(); it != tabla_procesos.end(); ++it) {
            if (it->ppid == pid_padre && it->estado == Estado::ZOMBIE) {
                std::cout << "  [WAIT] Padre PID " << pid_padre
                          << " recolectó estado del Hijo ZOMBIE PID " << it->pid
                          << ". Eliminando BCP de la tabla.\n";
                tabla_procesos.erase(it);
                return;
            }
        }
    }

    // Simulación de adopción de huérfanos por init (PID 1)
    void atender_huerfanos(int pid_padre_muerto) {
        for (auto& proc : tabla_procesos) {
            if (proc.ppid == pid_padre_muerto) {
                std::cout << "  [ADOPCIÓN] Proceso Huérfano PID " << proc.pid
                          << " fue adoptado por init (PID 1)\n";
                proc.ppid = 1;
            }
        }
    }

    BCP* obtenerBCP(int pid) {
        for (auto& proc : tabla_procesos) {
            if (proc.pid == pid) return &proc;
        }
        return nullptr;
    }

    void mostrarTablaProcesos() const {
        std::cout << "\n=================================================================================\n";
        std::cout << "               ESTADO ACTUAL DE LA TABLA DE PROCESOS DEL SISTEMA                 \n";
        std::cout << "=================================================================================\n";
        for (const auto& proc : tabla_procesos) {
            proc.imprimir();
        }
        std::cout << "=================================================================================\n\n";
    }
};

int main() {
    SistemaOperativoSimulado so;
    so.mostrarTablaProcesos();

    std::cout << "--- 1. Creación de Jerarquía con fork() ---\n";
    int pid_bash = so.simular_fork(1, "bash");
    int pid_gcc  = so.simular_fork(pid_bash, "gcc");
    so.mostrarTablaProcesos();

    std::cout << "--- 2. Reemplazo de Imagen con exec() ---\n";
    so.simular_exec(pid_gcc, "cc1_compiler");
    so.mostrarTablaProcesos();

    std::cout << "--- 3. Generación de Proceso Zombie ---\n";
    so.simular_exit(pid_gcc, 0); // gcc termina pero bash no ha hecho wait
    so.mostrarTablaProcesos();

    std::cout << "--- 4. Recolección de Zombie con wait() ---\n";
    so.simular_wait(pid_bash);
    so.mostrarTablaProcesos();

    std::cout << "--- 5. Simulación de Proceso Huérfano ---\n";
    int pid_worker = so.simular_fork(pid_bash, "worker_task");
    so.simular_exit(pid_bash, 0); // Muere el padre bash antes que worker_task
    so.atender_huerfanos(pid_bash);
    so.mostrarTablaProcesos();

    return 0;
}
```

In [ ]:
%%writefile laboratorio_jerarquia_bcp.cpp
// ==============================================================================
// EJERCICIO PRÁCTICO: JERARQUÍA DE PROCESOS Y CAMBIO DE CONTEXTO
// Lenguaje: C++17
// Asignatura: Sistemas Operativos - Unidad II: Administración de Procesos
// ==============================================================================

# include <iostream>
# include <vector>
# include <string>
# include <iomanip>
# include <memory>

enum class Estado { NUEVO, LISTO, EJECUTANDO, BLOQUEADO, ZOMBIE, TERMINADO };

std::string estadoToString(Estado e) {
    switch (e) {
        case Estado::NUEVO:      return "NUEVO";
        case Estado::LISTO:      return "LISTO";
        case Estado::EJECUTANDO: return "EJECUTANDO";
        case Estado::BLOQUEADO:  return "BLOQUEADO";
        case Estado::ZOMBIE:     return "ZOMBIE";
        case Estado::TERMINADO:  return "TERMINADO";
        default:                 return "DESCONOCIDO";
    }
}

// Estructura del Bloque de Control de Proceso (BCP)
struct BCP {
    int pid;
    int ppid;
    std::string nombre_programa;
    Estado estado;
    uint16_t PC; // Program Counter
    uint16_t SP; // Stack Pointer
    int PSW_modo; // 0 = Usuario, 1 = Kernel
    int codigo_salida;

    void imprimir() const {
        std::cout << "[BCP PID=" << std::setw(3) << pid
                  << " | PPID=" << std::setw(3) << ppid
                  << " | Prog: " << std::left << std::setw(12) << nombre_programa
                  << " | Estado: " << std::setw(10) << estadoToString(estado)
                  << " | PC: 0x" << std::hex << std::uppercase << std::setw(4) << std::setfill('0') << PC << std::dec
                  << " | Modo: " << (PSW_modo == 1 ? "KERNEL" : "USUARIO") << "]\n";
    }
};

// Tabla de Procesos Simulada del Sistema Operativo
class SistemaOperativoSimulado {
private:
    std::vector<BCP> tabla_procesos;
    int contador_pid = 100;
    int pid_activo = -1;

public:
    SistemaOperativoSimulado() {
        // Crear proceso raíz (init - PID 1)
        BCP init{1, 0, "init", Estado::EJECUTANDO, 0x0010, 0x0FFF, 1, 0};
        tabla_procesos.push_back(init);
        pid_activo = 1;
    }

    // Simulación de la llamada fork()
    int simular_fork(int pid_padre, const std::string& nombre_hijo) {
        BCP* padre = obtenerBCP(pid_padre);
        if (!padre) return -1;

        contador_pid++;
        BCP hijo;
        hijo.pid = contador_pid;
        hijo.ppid = padre->pid;
        hijo.nombre_programa = nombre_hijo;
        hijo.estado = Estado::LISTO;
        hijo.PC = padre->PC; // Copia el PC del padre (misma instrucción)
        hijo.SP = padre->SP - 0x0100; // Nueva pila
        hijo.PSW_modo = 0;
        hijo.codigo_salida = 0;

        tabla_procesos.push_back(hijo);
        std::cout << "  [FORK] Proceso Padre PID " << pid_padre
                  << " creó al Proceso Hijo PID " << hijo.pid << " (" << nombre_hijo << ")\n";
        return hijo.pid;
    }

    // Simulación de la llamada exec()
    bool simular_exec(int pid, const std::string& nuevo_programa) {
        BCP* proc = obtenerBCP(pid);
        if (!proc) return false;

        std::cout << "  [EXEC] Proceso PID " << pid << " reemplaza su imagen por '" << nuevo_programa << "'\n";
        proc->nombre_programa = nuevo_programa;
        proc->PC = 0x1000; // Reinicia PC al punto de entrada del nuevo ejecutable
        return true;
    }

    // Simulación de terminación exit() y estado Zombie
    void simular_exit(int pid, int status) {
        BCP* proc = obtenerBCP(pid);
        if (!proc) return;

        proc->codigo_salida = status;
        proc->estado = Estado::ZOMBIE;
        std::cout << "  [EXIT] Proceso PID " << pid << " finalizó con código " << status
                  << " -> Entra a Estado ZOMBIE (esperando wait del padre)\n";
    }

    // Simulación de la llamada wait() recolectora
    void simular_wait(int pid_padre) {
        for (auto it = tabla_procesos.begin(); it != tabla_procesos.end(); ++it) {
            if (it->ppid == pid_padre && it->estado == Estado::ZOMBIE) {
                std::cout << "  [WAIT] Padre PID " << pid_padre
                          << " recolectó estado del Hijo ZOMBIE PID " << it->pid
                          << ". Eliminando BCP de la tabla.\n";
                tabla_procesos.erase(it);
                return;
            }
        }
    }

    // Simulación de adopción de huérfanos por init (PID 1)
    void atender_huerfanos(int pid_padre_muerto) {
        for (auto& proc : tabla_procesos) {
            if (proc.ppid == pid_padre_muerto) {
                std::cout << "  [ADOPCIÓN] Proceso Huérfano PID " << proc.pid
                          << " fue adoptado por init (PID 1)\n";
                proc.ppid = 1;
            }
        }
    }

    BCP* obtenerBCP(int pid) {
        for (auto& proc : tabla_procesos) {
            if (proc.pid == pid) return &proc;
        }
        return nullptr;
    }

    void mostrarTablaProcesos() const {
        std::cout << "\n=================================================================================\n";
        std::cout << "               ESTADO ACTUAL DE LA TABLA DE PROCESOS DEL SISTEMA                 \n";
        std::cout << "=================================================================================\n";
        for (const auto& proc : tabla_procesos) {
            proc.imprimir();
        }
        std::cout << "=================================================================================\n\n";
    }
};

int main() {
    SistemaOperativoSimulado so;
    so.mostrarTablaProcesos();

    std::cout << "--- 1. Creación de Jerarquía con fork() ---\n";
    int pid_bash = so.simular_fork(1, "bash");
    int pid_gcc  = so.simular_fork(pid_bash, "gcc");
    so.mostrarTablaProcesos();

    std::cout << "--- 2. Reemplazo de Imagen con exec() ---\n";
    so.simular_exec(pid_gcc, "cc1_compiler");
    so.mostrarTablaProcesos();

    std::cout << "--- 3. Generación de Proceso Zombie ---\n";
    so.simular_exit(pid_gcc, 0); // gcc termina pero bash no ha hecho wait
    so.mostrarTablaProcesos();

    std::cout << "--- 4. Recolección de Zombie con wait() ---\n";
    so.simular_wait(pid_bash);
    so.mostrarTablaProcesos();

    std::cout << "--- 5. Simulación de Proceso Huérfano ---\n";
    int pid_worker = so.simular_fork(pid_bash, "worker_task");
    so.simular_exit(pid_bash, 0); // Muere el padre bash antes que worker_task
    so.atender_huerfanos(pid_bash);
    so.mostrarTablaProcesos();

    return 0;
}

Writing laboratorio_jerarquia_bcp.cpp


### [CÓDIGO 3.3] Compilación y Ejecución del código
```c++
# Compilar en C++17 con g++
!g++ -O3 -std=c++17 laboratorio_jerarquia_bcp.cpp -o laboratorio_jerarquia_bcp

# Ejecutar la simulación
!./laboratorio_jerarquia_bcp
```

In [ ]:
# Compilar en C++17 con g++
!g++ -O3 -std=c++17 laboratorio_jerarquia_bcp.cpp -o laboratorio_jerarquia_bcp

# Ejecutar la simulación
!./laboratorio_jerarquia_bcp


               ESTADO ACTUAL DE LA TABLA DE PROCESOS DEL SISTEMA                 
[BCP PID=  1 | PPID=  0 | Prog: init         | Estado: EJECUTANDO | PC: 0x1000 | Modo: KERNEL]

--- 1. Creación de Jerarquía con fork() ---
  [FORK] Proceso Padre PID 1 creó al Proceso Hijo PID 101 (bash)
  [FORK] Proceso Padre PID 101 creó al Proceso Hijo PID 102 (gcc)

               ESTADO ACTUAL DE LA TABLA DE PROCESOS DEL SISTEMA                 
[BCP PID=100 | PPID=000 | Prog: init00000000 | Estado: EJECUTANDO | PC: 0x1000 | Modo: KERNEL]
[BCP PID=101 | PPID=100 | Prog: bash00000000 | Estado: LISTO00000 | PC: 0x1000 | Modo: USUARIO]
[BCP PID=102 | PPID=101 | Prog: gcc000000000 | Estado: LISTO00000 | PC: 0x1000 | Modo: USUARIO]

--- 2. Reemplazo de Imagen con exec() ---
  [EXEC] Proceso PID 102 reemplaza su imagen por 'cc1_compiler'

               ESTADO ACTUAL DE LA TABLA DE PROCESOS DEL SISTEMA                 
[BCP PID=100 | PPID=000 | Prog: init00000000 | Estado: EJECUTANDO | PC: 0x1000 | Modo:

## 4. Sección Práctica 3: Inter Process Communication (IPC).  

En esta sección implementarás y analizarás los mecanismos de Comunicación entre Procesos (IPC) y sincronización concurrente en sistemas operativos mediante la inspección de recursos del kernel en Linux y la resolución del problema clásico del Productor-Consumidor usando semáforos POSIX y tuberías en C++17.

### [CÓDIGO 4.1] Inspección de Mecanismos de IPC en el Entorno Linux.

Ejecutar los siguientes comandos en celdas de código bash para inspeccionar las facilidades de IPC del kernel Linux:
```bash
# 1. Consultar recursos de IPC activos en el sistema (Memoria compartida, Semáforos, Colas de mensajes)
!ipcs -a

# 2. Consultar el límite máximo del tamaño del buffer para tuberías en Linux (/proc)
!cat /proc/sys/fs/pipe-max-size

# 3. Crear y probar una tubería con nombre (FIFO) en la línea de comandos de Linux
!mkfifo /tmp/mi_tubería_fifo
!ls -l /tmp/mi_tubería_fifo
!rm /tmp/mi_tubería_fifo
```

In [ ]:
# 1. Consultar recursos de IPC activos en el sistema (Memoria compartida, Semáforos, Colas de mensajes)
!ipcs -a

# 2. Consultar el límite máximo del tamaño del buffer para tuberías en Linux (/proc)
!cat /proc/sys/fs/pipe-max-size

# 3. Crear y probar una tubería con nombre (FIFO) en la línea de comandos de Linux
!mkfifo /tmp/mi_tubería_fifo
!ls -l /tmp/mi_tubería_fifo
!rm /tmp/mi_tubería_fifo


------ Message Queues --------
key        msqid      owner      perms      used-bytes   messages    

------ Shared Memory Segments --------
key        shmid      owner      perms      bytes      nattch     status      

------ Semaphore Arrays --------
key        semid      owner      perms      nsems     

1048576
prw-r--r-- 1 root root 0 Oct 21 18:36 /tmp/mi_tubería_fifo


### [CÓDIGO 4.2] Simulación del Productor-Consumidor con Semáforos POSIX y Threads.

Copiar y ejecutar el siguiente código en Colab:
```cpp
%%writefile laboratorio_ipc_semaphoro.cpp
// ==============================================================================
// EJERCICIO PRÁCTICO: IPC Y SINCRONIZACIÓN CON SEMÁFOROS POSIX Y TUBERÍAS
// Lenguaje: C++17 (POSIX Threads & Semaphores)
// Asignatura: Sistemas Operativos - Unidad II: Administración de Procesos
// ==============================================================================

#include <iostream>
#include <thread>
#include <vector>
#include <chrono>
#include <semaphore.h>
#include <fcntl.h>
#include <unistd.h>
#include <sys/wait.h>

constexpr int TAMANO_BUFFER = 5;
constexpr int TOTAL_ITEMS = 10;

// Buffer circular compartido
int buffer[TAMANO_BUFFER];
int in_idx = 0;
int out_idx = 0;

// Semáforos POSIX
sem_t sem_huecos;    // Cuenta espacios libres en el buffer
sem_t sem_elementos; // Cuenta elementos disponibles para consumir
sem_t sem_mutex;     // Garantiza exclusión mutua (semáforo binario)

// Función del Proceso/Hilo Productor
void productor() {
    for (int i = 1; i <= TOTAL_ITEMS; ++i) {
        // 1. Producir dato
        int dato = i * 10;

        // 2. Esperar por un hueco libre
        sem_wait(&sem_huecos);

        // 3. Entrar a la región crítica (Mutex Down)
        sem_wait(&sem_mutex);

        // --- REGIÓN CRÍTICA ---
        buffer[in_idx] = dato;
        std::cout << "  [PRODUCTOR] Produjo e insertó item " << dato
                  << " en la posición " << in_idx << "\n";
        in_idx = (in_idx + 1) % TAMANO_BUFFER;
        // ----------------------

        // 4. Salir de la región crítica (Mutex Up)
        sem_post(&sem_mutex);

        // 5. Notificar que hay un nuevo elemento disponible
        sem_post(&sem_elementos);

        std::this_thread::sleep_for(std::chrono::milliseconds(150));
    }
}

// Función del Proceso/Hilo Consumidor
void consumidor() {
    for (int i = 1; i <= TOTAL_ITEMS; ++i) {
        // 1. Esperar a que exista al menos un elemento
        sem_wait(&sem_elementos);

        // 2. Entrar a la región crítica (Mutex Down)
        sem_wait(&sem_mutex);

        // --- REGIÓN CRÍTICA ---
        int dato = buffer[out_idx];
        std::cout << "    [CONSUMIDOR] Extrajo y consumió item " << dato
                  << " de la posición " << out_idx << "\n";
        out_idx = (out_idx + 1) % TAMANO_BUFFER;
        // ----------------------

        // 3. Salir de la región crítica (Mutex Up)
        sem_post(&sem_mutex);

        // 4. Notificar que se liberó un hueco
        sem_post(&sem_huecos);

        std::this_thread::sleep_for(std::chrono::milliseconds(300));
    }
}

// Función de demostración de Tubería Unidireccional (Pipe) entre procesos con fork()
void probar_tubería_pipe() {
    std::cout << "\n=================================================================\n";
    std::cout << "   DEMOSTRACIÓN DE COMUNICACIÓN POR TUBERÍA (PIPE EN LINUX)      \n";
    std::cout << "=================================================================\n";

    int fd[1]; // fd = lectura, fd[2] = escritura
    if (pipe(fd) == -1) {
        std::cerr << "Error al crear la tubería\n";
        return;
    }

    pid_t pid = fork();

    if (pid < 0) {
        std::cerr << "Error en fork()\n";
        return;
    }

    if (pid == 0) {
        // --- PROCESO HIJO (Escritor en la Tubería) ---
        close(fd); // Cierra el extremo de lectura que no utiliza
        std::string mensaje = "¡Saludos desde el Proceso Hijo mediante Pipe POSIX!";
        std::cout << "  [PROCESO HIJO] Enviando mensaje por el extremo de escritura fd[2]...\n";
        write(fd[2], mensaje.c_str(), mensaje.length() + 1);
        close(fd[2]); // Cierra escritura para enviar EOF
        _exit(0);
    } else {
        // --- PROCESO PADRE (Lector de la Tubería) ---
        close(fd[2]); // Cierra el extremo de escritura que no utiliza
        char buffer_lectura[3];
        std::cout << "  [PROCESO PADRE] Esperando lectura en el extremo fd (Lectura Bloqueante)...\n";
        read(fd, buffer_lectura, sizeof(buffer_lectura));
        std::cout << "  [PROCESO PADRE] Mensaje recibido del Hijo: \"" << buffer_lectura << "\"\n";
        close(fd);
        wait(NULL); // Recolecta proceso hijo
    }
}

int main() {
    std::cout << "=================================================================\n";
    std::cout << "   SIMULACIÓN DEL PROBLEMA PRODUCTOR-CONSUMIDOR CON SEMÁFOROS    \n";
    std::cout << "=================================================================\n\n";

    // Inicialización de Semáforos POSIX
    sem_init(&sem_huecos, 0, TAMANO_BUFFER); // Inicializado en N (5)
    sem_init(&sem_elementos, 0, 0);          // Inicializado en 0
    sem_init(&sem_mutex, 0, 1);              // Inicializado en 1 (Mutex binario)

    // Crear hilos concurrentes para Productor y Consumidor
    std::thread hilo_productor(productor);
    std::thread hilo_consumidor(consumidor);

    // Esperar finalización de hilos
    hilo_productor.join();
    hilo_consumidor.join();

    // Destruir semáforos
    sem_destroy(&sem_huecos);
    sem_destroy(&sem_elementos);
    sem_destroy(&sem_mutex);

    // Ejecutar demostración de tuberías entre procesos
    probar_tubería_pipe();

    return 0;
}
```

In [ ]:
%%writefile laboratorio_ipc_semaphoro.cpp
// ==============================================================================
// EJERCICIO PRÁCTICO: IPC Y SINCRONIZACIÓN CON SEMÁFOROS POSIX Y TUBERÍAS
// Lenguaje: C++17 (POSIX Threads & Semaphores)
// Asignatura: Sistemas Operativos - Unidad II: Administración de Procesos
// ==============================================================================

# include <iostream>
# include <thread>
# include <vector>
# include <chrono>
# include <semaphore.h>
# include <fcntl.h>
# include <unistd.h>
# include <sys/wait.h>

constexpr int TAMANO_BUFFER = 5;
constexpr int TOTAL_ITEMS = 10;

// Buffer circular compartido
int buffer[TAMANO_BUFFER];
int in_idx = 0;
int out_idx = 0;

// Semáforos POSIX
sem_t sem_huecos;    // Cuenta espacios libres en el buffer
sem_t sem_elementos; // Cuenta elementos disponibles para consumir
sem_t sem_mutex;     // Garantiza exclusión mutua (semáforo binario)

// Función del Proceso/Hilo Productor
void productor() {
    for (int i = 1; i <= TOTAL_ITEMS; ++i) {
        // 1. Producir dato
        int dato = i * 10;

        // 2. Esperar por un hueco libre
        sem_wait(&sem_huecos);

        // 3. Entrar a la región crítica (Mutex Down)
        sem_wait(&sem_mutex);

        // --- REGIÓN CRÍTICA ---
        buffer[in_idx] = dato;
        std::cout << "  [PRODUCTOR] Produjo e insertó item " << dato
                  << " en la posición " << in_idx << "\n";
        in_idx = (in_idx + 1) % TAMANO_BUFFER;
        // ----------------------

        // 4. Salir de la región crítica (Mutex Up)
        sem_post(&sem_mutex);

        // 5. Notificar que hay un nuevo elemento disponible
        sem_post(&sem_elementos);

        std::this_thread::sleep_for(std::chrono::milliseconds(150));
    }
}

// Función del Proceso/Hilo Consumidor
void consumidor() {
    for (int i = 1; i <= TOTAL_ITEMS; ++i) {
        // 1. Esperar a que exista al menos un elemento
        sem_wait(&sem_elementos);

        // 2. Entrar a la región crítica (Mutex Down)
        sem_wait(&sem_mutex);

        // --- REGIÓN CRÍTICA ---
        int dato = buffer[out_idx];
        std::cout << "    [CONSUMIDOR] Extrajo y consumió item " << dato
                  << " de la posición " << out_idx << "\n";
        out_idx = (out_idx + 1) % TAMANO_BUFFER;
        // ----------------------

        // 3. Salir de la región crítica (Mutex Up)
        sem_post(&sem_mutex);

        // 4. Notificar que se liberó un hueco
        sem_post(&sem_huecos);

        std::this_thread::sleep_for(std::chrono::milliseconds(300));
    }
}

// Función de demostración de Tubería Unidireccional (Pipe) entre procesos con fork()
void probar_tubería_pipe() {
    std::cout << "\n=================================================================\n";
    std::cout << "   DEMOSTRACIÓN DE COMUNICACIÓN POR TUBERÍA (PIPE EN LINUX)      \n";
    std::cout << "=================================================================\n";

    int fd[1]; // fd = lectura, fd[2] = escritura
    if (pipe(fd) == -1) {
        std::cerr << "Error al crear la tubería\n";
        return;
    }

    pid_t pid = fork();

    if (pid < 0) {
        std::cerr << "Error en fork()\n";
        return;
    }

    if (pid == 0) {
        // --- PROCESO HIJO (Escritor en la Tubería) ---
        close(fd); // Cierra el extremo de lectura que no utiliza
        std::string mensaje = "¡Saludos desde el Proceso Hijo mediante Pipe POSIX!";
        std::cout << "  [PROCESO HIJO] Enviando mensaje por el extremo de escritura fd[2]...\n";
        write(fd[2], mensaje.c_str(), mensaje.length() + 1);
        close(fd[2]); // Cierra escritura para enviar EOF
        _exit(0);
    } else {
        // --- PROCESO PADRE (Lector de la Tubería) ---
        close(fd[2]); // Cierra el extremo de escritura que no utiliza
        char buffer_lectura[3];
        std::cout << "  [PROCESO PADRE] Esperando lectura en el extremo fd (Lectura Bloqueante)...\n";
        read(fd, buffer_lectura, sizeof(buffer_lectura));
        std::cout << "  [PROCESO PADRE] Mensaje recibido del Hijo: \"" << buffer_lectura << "\"\n";
        close(fd);
        wait(NULL); // Recolecta proceso hijo
    }
}

int main() {
    std::cout << "=================================================================\n";
    std::cout << "   SIMULACIÓN DEL PROBLEMA PRODUCTOR-CONSUMIDOR CON SEMÁFOROS    \n";
    std::cout << "=================================================================\n\n";

    // Inicialización de Semáforos POSIX
    sem_init(&sem_huecos, 0, TAMANO_BUFFER); // Inicializado en N (5)
    sem_init(&sem_elementos, 0, 0);          // Inicializado en 0
    sem_init(&sem_mutex, 0, 1);              // Inicializado en 1 (Mutex binario)

    // Crear hilos concurrentes para Productor y Consumidor
    std::thread hilo_productor(productor);
    std::thread hilo_consumidor(consumidor);

    // Esperar finalización de hilos
    hilo_productor.join();
    hilo_consumidor.join();

    // Destruir semáforos
    sem_destroy(&sem_huecos);
    sem_destroy(&sem_elementos);
    sem_destroy(&sem_mutex);

    // Ejecutar demostración de tuberías entre procesos
    probar_tubería_pipe();

    return 0;
}

Writing laboratorio_ipc_semaphoro.cpp


### [CÓDIGO 4.3] Compilación y Ejecución del código.

```bash
# Compilar en C++17 con g++ habilitando bibliotecas de hilos y sincronización (-pthread)
!g++ -O3 -std=c++17 laboratorio_ipc_semaphoro.cpp -o laboratorio_ipc_semaphoro -pthread

# Ejecutar la simulación completa de IPC
!./laboratorio_ipc_semaphoro
```

In [ ]:
# Compilar en C++17 con g++ habilitando bibliotecas de hilos y sincronización (-pthread)
!g++ -O3 -std=c++17 laboratorio_ipc_semaphoro.cpp -o laboratorio_ipc_semaphoro -pthread

# Ejecutar la simulación completa de IPC
!./laboratorio_ipc_semaphoro

laboratorio_ipc_semaphoro.cpp: In function ‘void probar_tubería_pipe()’:
laboratorio_ipc_semaphoro.cpp:105:15: error: invalid conversion from ‘int*’ to ‘int’ []8;;https://gcc.gnu.org/onlinedocs/gcc/Warning-Options.html#index-fpermissive-fpermissive]8;;]
  105 |         close(fd); // Cierra el extremo de lectura que no utiliza
      |               ^~
      |               |
      |               int*
In file included from laboratorio_ipc_semaphoro.cpp:13:
/usr/include/unistd.h:358:23: note:   initializing argument 1 of ‘int close(int)’
  358 | extern int close (int __fd);
      |                   ~~~~^~~~
laboratorio_ipc_semaphoro.cpp:116:14: error: invalid conversion from ‘int*’ to ‘int’ []8;;https://gcc.gnu.org/onlinedocs/gcc/Warning-Options.html#index-fpermissive-fpermissive]8;;]
  116 |         read(fd, buffer_lectura, sizeof(buffer_lectura));
      |              ^~
      |              |
      |              int*
In file included from /usr/include/unistd.h:1217:
/usr/inc

## 6. Desafío Asíncrono / Entregable de la Unidad I

### Extensión y Análisis de la Tabla de Procesos y el BCP
A partir del laboratorio realizado en Google Colab sobre inspección de procesos en Linux y la estructura struct BCP en C++17, resuelve de manera individual los siguientes ítems modificando y extendiendo el código fuente base.

#### Instrucciones::
1. Análisis de Procesos del Sistema en Linux (/proc)
Contexto: El sistema de archivos virtual /proc contiene directorios nombrados con el PID de cada proceso activo, exponiendo archivos de control e información detallada del kernel.  
**Consigna:**  
> Ejecuta en una celda de Colab un comando o un pequeño script en Python/Bash que busque e imprima el uso de memoria actual y el estado de ejecución (campo State) de tres procesos distintos del sistema leyendo directamente sus archivos dentro de /proc/[PID]/status.  
Explica brevemente en un párrafo por qué el directorio /proc no ocupa espacio real en el disco duro de almacenamiento masivo.

2. Ampliación del Modelo de Datos del BCP en C++17
Contexto: Un sistema operativo real almacena más información de gestión en el Bloque de Control de Proceso, especialmente relacionada con la contabilidad de tiempo y la administración de archivos abiertos.  
**Consigna:**  
>Modifica el archivo bcp_struct.cpp para incorporar los siguientes campos dentro de la struct BCP:  
uint64_t tiempo_cpu_ms; (Tiempo total de CPU consumido en milisegundos).  
uint8_t archivos_abiertos; (Número de descriptores de archivos o sockets abiertos por el proceso).  
Actualiza el método mostrarBCP() para que despliegue formateados estos nuevos atributos en la ficha técnica de la consola.

3. Simulación de Transición de Estados
Contexto: Los procesos cambian dinámicamente de estado en función de los eventos del sistema y la planificación del kernel (Modelo de 5 Estados).  
**Consigna:**  
>Implementa una función en C++ dentro del programa que reciba por referencia un objeto BCP y un evento (por ejemplo, una interrupción por fin de E/S o una expiración de cuanto de tiempo/quantum), de modo que simule la transición de estado (por ejemplo, pasando de BLOQUEADO a LISTO) e imprima un mensaje descriptivo del cambio ocurrido.

### Gestión Avanzada y Ciclo de Vida de Procesos.  

#### Instrucciones:
A partir del laboratorio de jerarquía de procesos y simulación del BCP, resuelve de manera individual los siguientes ítems ampliando el código fuente de laboratorio_jerarquia_bcp.cpp.

1. Análisis de Procesos Zombie en Sistemas Linux Reales
Contexto: En los sistemas operativos tipo Unix, un proceso zombie ocurre cuando un hijo termina su ejecución, pero su padre aún no recolecta su estado de salida mediante la llamada wait().  
**Consigna:**  
> Investiga y redacta brevemente cómo se visualiza un proceso en estado zombie utilizando comandos de consola en Linux (por ejemplo, el indicador Z en ps aux o top).  
Responde en un párrafo de análisis: ¿Por qué el kernel de Linux mantiene el Bloque de Control de Proceso (BCP/Task Struct) en memoria para un proceso que ya ha terminado de ejecutarse y liberado su código en disco?  

2. Automatización de la Adopción de Huérfanos
Contexto: En el código base actual, la adopción de procesos huérfanos requiere invocar de manera explícita el método atender_huerfanos(). En un sistema operativo real, el kernel gestiona esto de forma automática cuando un proceso padre muere.  
**Consigna:**
> Modifica la lógica dentro de la clase SistemaOperativoSimulado para que, al ejecutar el método simular_exit(), el sistema detecte de forma automática si el proceso que termina tenía hijos activos y reasigne de inmediato su ppid al proceso init (PID 1), eliminando la necesidad de llamar al método de huérfanos por separado en el programa principal main.  

3. Extensión del Modelo de Estados (Estado Suspendido)
Contexto: Los modelos de procesos reales suelen incorporar más estados para manejar suspensiones temporales por intervención del usuario o del sistema (por ejemplo, mediante señales como SIGSTOP y SIGCONT).  
**Consigna:**  
> Agrega el estado SUSPENDIDO a la enumeración Estado.  
Implementa un nuevo método en el simulador llamado simular_suspension(int pid) que permita cambiar el estado de un proceso a SUSPENDIDO y guarde una traza en consola describiendo el cambio de contexto realizado.

### Concurrencia Avanzada y Robustez en IPC.  


A partir del laboratorio de mecanismos de IPC y sincronización, resuelve de manera individual los siguientes ítems ampliando y depurando el código fuente base.
#### Instrucciones:  

1. Auditoría de Recursos IPC en Linux.  
Contexto: Los recursos de IPC creados por los programas en sistemas Unix deben ser gestionados cuidadosamente por el kernel para evitar fugas de recursos (resource leaks).  
**Consigna:**  
> Ejecuta en una celda de Google Colab los comandos ipcs -m (para memoria compartida) e ipcs -s (para semáforos del sistema) y captura su salida.  
Explica brevemente en un párrafo qué representan las columnas clave (key), identificador (shmid/semid) y propietario (owner) que muestra el sistema operativo.

2. Escalabilidad a Múltiples Productores y Consumidores.  
Contexto: En escenarios reales de sistemas operativos, múltiples hilos o procesos intentan acceder concurrentemente a los recursos compartidos.  
**Consigna:**
> Modifica la función main() del código base para instanciar dos hilos productores y dos hilos consumidores de forma simultánea. Asegúrate de verificar que el semáforo de exclusión mutua (sem_mutex) y los contadores de huecos/elementos previenen eficazmente las condiciones de carrera y la corrupción del buffer circular.

3. Depuración y Análisis de Descriptores en Tuberías (Pipe).  
Contexto: La correcta gestión de los descriptores de archivos (file descriptors) es vital en las tuberías para evitar bloqueos permanentes (deadlocks) por lecturas colgadas en la espera de un fin de archivo (EOF).  
**Consigna:**  
> Analiza la función probar_tubería_pipe(). Identifica por qué el manejo de los descriptores de archivo requiere un arreglo de tamaño 2 (int fd[2]) para separar adecuadamente el canal de lectura (fd) del canal de escritura (fd), y documenta qué ocurre si el proceso padre olvida cerrar el extremo de escritura antes de realizar la operación de lectura bloqueante.
---

# Desafío Asíncrono Contestado

## Extensión y Análisis de la Tabla de Procesos y el BCP

### 1.- Análisis de Procesos Zombie en Sistemas Linux Reales Contexto: En los sistemas operativos tipo Unix, un proceso zombie ocurre cuando un hijo termina su ejecución, pero su padre aún no recolecta su estado de salida mediante la llamada wait().

In [2]:
import os

def obtener_info_proceso(pid):
    ruta = f"/proc/{pid}/status"
    info = {"Name": "N/D", "State": "N/D", "VmRSS": "N/D", "VmSize": "N/D"}
    with open(ruta, "r", encoding="utf-8", errors="replace") as archivo:
        for linea in archivo:
            if linea.startswith(("Name:", "State:", "VmRSS:", "VmSize:")):
                clave, _, valor = linea.partition(":")
                info[clave] = valor.strip()
    return info

# Seleccionamos 3 procesos distintos: PID 1, proceso padre actual y proceso actual
candidatos = [1, os.getppid(), os.getpid()]
pids = []
for p in candidatos:
    if p not in pids and os.path.exists(f"/proc/{p}/status"):
        pids.append(p)

# Rellenar hasta 3 si alguno se repite o no aplica
for entrada in sorted(os.listdir("/proc"), key=lambda x: int(x) if x.isdigit() else -1):
    if len(pids) >= 3:
        break
    if entrada.isdigit():
        p = int(entrada)
        if p not in pids and os.path.exists(f"/proc/{p}/status"):
            pids.append(p)

print(f"{'PID':<7} {'Proceso':<18} {'State':<20} {'Memoria Física (VmRSS)':<24} {'Memoria Virtual (VmSize)'}")
print("-" * 92)
for p in pids:
    d = obtener_info_proceso(p)
    print(f"{p:<7} {d['Name']:<18} {d['State']:<20} {d['VmRSS']:<24} {d['VmSize']}")

PID     Proceso            State                Memoria Física (VmRSS)   Memoria Virtual (VmSize)
--------------------------------------------------------------------------------------------
1       docker-init        S (sleeping)         564 kB                   984 kB
103     jupyter-server     R (running)          162248 kB                590392 kB
316     python3            R (running)          105052 kB                690068 kB


El directorio /proc no ocupa espacio real en el disco duro porque es un sistema de archivos virtual (procfs) gestionado en memoria por el kernel de Linux. Sus archivos y directorios no se almacenan como bloques de datos en un dispositivo de almacenamiento persistente, sino que el kernel los sintetiza al vuelo cada vez que un proceso los consulta a partir de sus estructuras internas en RAM (como el task_struct o Bloque de Control de Proceso). Por ello su contenido refleja el estado instantáneo del sistema y desaparece o cambia dinámicamente sin consumir espacio en disco.

### 2.- Ampliación del Modelo de Datos del BCP en C++17 Contexto: Un sistema operativo real almacena más información de gestión en el Bloque de Control de Proceso, especialmente relacionada con la contabilidad de tiempo y la administración de archivos abiertos.

In [3]:
%%writefile bcp_struct.cpp
#include <iostream>
#include <string>
#include <vector>
#include <iomanip>
#include <cstdint>
#include <sstream>

enum class EstadoProceso {
    NUEVO,
    LISTO,
    EJECUTANDO,
    BLOQUEADO,
    SALIENTE
};

std::string estadoToString(EstadoProceso estado) {
    switch (estado) {
        case EstadoProceso::NUEVO:      return "NUEVO";
        case EstadoProceso::LISTO:      return "LISTO";
        case EstadoProceso::EJECUTANDO: return "EJECUTANDO";
        case EstadoProceso::BLOQUEADO:  return "BLOQUEADO";
        case EstadoProceso::SALIENTE:   return "SALIENTE";
        default:                        return "DESCONOCIDO";
    }
}

static size_t longitudUTF8(const std::string& s) {
    size_t n = 0;
    for (unsigned char c : s) {
        if ((c & 0xC0) != 0x80) ++n;
    }
    return n;
}

static std::string hexFmt(uint32_t valor, int digitos) {
    std::ostringstream o;
    o << "0x" << std::hex << std::uppercase << std::setfill('0') << std::setw(digitos) << valor;
    return o.str();
}

static void imprimirLinea(const std::string& texto) {
    const size_t ANCHO = 65;
    size_t len = longitudUTF8(texto);
    std::cout << "│" << texto << std::string(len < ANCHO ? ANCHO - len : 0, ' ') << "│\n";
}

struct BCP {
    // 1. Identificadores
    uint32_t pid;
    uint32_t ppid;
    std::string nombre_programa;

    // 2. Información de Estado del Procesador
    EstadoProceso estado_actual;
    uint16_t PC;
    uint16_t SP;
    int AC;
    uint8_t PSW_modo;

    // 3. Gestión de Memoria y Planificación
    uint32_t direccion_base_ram;
    uint32_t limite_memoria_bytes;
    int prioridad;

    // 4. Contabilidad de CPU y descriptores de archivos
    uint64_t tiempo_cpu_ms;
    uint8_t archivos_abiertos;

    void mostrarBCP() const {
        std::ostringstream l;

        std::cout << "┌" << std::string(65, '-') << "┐\n";
        l << " FICHA TÉCNICA DEL BCP (PID: " << std::setw(5) << pid << ") - Programa: " << nombre_programa;
        imprimirLinea(l.str()); l.str("");

        std::cout << "├" << std::string(65, '-') << "┤\n";
        l << " Estado Actual  : " << std::left << std::setw(12) << estadoToString(estado_actual)
          << " | PPID Padre     : " << ppid;
        imprimirLinea(l.str()); l.str("");

        l << " Contexto CPU   : PC=" << hexFmt(PC, 4) << " | SP=" << hexFmt(SP, 4) << " | AC=" << AC;
        imprimirLinea(l.str()); l.str("");

        l << " Modo PSW       : " << (PSW_modo == 1 ? "KERNEL (1)" : "USUARIO (0)")
          << " | Prioridad      : " << prioridad;
        imprimirLinea(l.str()); l.str("");

        l << " Memoria Asign. : Base=" << hexFmt(direccion_base_ram, 6)
          << " | Límite=" << limite_memoria_bytes << " Bytes";
        imprimirLinea(l.str()); l.str("");

        l << " Contabilidad   : CPU=" << tiempo_cpu_ms << " ms"
          << " | Archivos abiertos=" << static_cast<int>(archivos_abiertos);
        imprimirLinea(l.str());

        std::cout << "└" << std::string(65, '-') << "┘\n";
    }
};

int main() {
    std::vector<BCP> tabla_procesos;

    BCP p1{101, 1, "chrome.exe", EstadoProceso::EJECUTANDO, 0x0420, 0x0FFF, 150, 0,
           0x10000, 65536, 5, 1250, 12};
    BCP p2{102, 101, "g++_compiler", EstadoProceso::BLOQUEADO, 0x08A2, 0x0E00, 0, 0,
           0x20000, 131072, 10, 340, 4};

    tabla_procesos.push_back(p1);
    tabla_procesos.push_back(p2);

    for (const auto& bcp : tabla_procesos) {
        bcp.mostrarBCP();
        std::cout << "\n";
    }
    return 0;
}

Writing bcp_struct.cpp


In [4]:
!g++ -O3 -std=c++17 bcp_struct.cpp -o bcp_struct
!./bcp_struct

┌-----------------------------------------------------------------┐
│ FICHA TÉCNICA DEL BCP (PID:   101) - Programa: chrome.exe       │
├-----------------------------------------------------------------┤
│ Estado Actual  : EJECUTANDO   | PPID Padre     : 1              │
│ Contexto CPU   : PC=0x0420 | SP=0x0FFF | AC=150                 │
│ Modo PSW       : USUARIO (0) | Prioridad      : 5               │
│ Memoria Asign. : Base=0x010000 | Límite=65536 Bytes             │
│ Contabilidad   : CPU=1250 ms | Archivos abiertos=12             │
└-----------------------------------------------------------------┘

┌-----------------------------------------------------------------┐
│ FICHA TÉCNICA DEL BCP (PID:   102) - Programa: g++_compiler     │
├-----------------------------------------------------------------┤
│ Estado Actual  : BLOQUEADO    | PPID Padre     : 101            │
│ Contexto CPU   : PC=0x08A2 | SP=0x0E00 | AC=0                   │
│ Modo PSW       : USUARIO (0) | Prioridad     

### 3.- Simulación de Transición de Estados Contexto: Los procesos cambian dinámicamente de estado en función de los eventos del sistema y la planificación del kernel (Modelo de 5 Estados).

In [5]:
%%writefile bcp_transiciones.cpp
#include <iostream>
#include <string>
#include <iomanip>
#include <cstdint>

enum class EstadoProceso {
    NUEVO,
    LISTO,
    EJECUTANDO,
    BLOQUEADO,
    SALIENTE
};

enum class EventoProceso {
    ADMITIR,
    DESPACHAR,
    FIN_QUANTUM,
    SOLICITAR_ES,
    FIN_ES,
    TERMINAR
};

struct BCP {
    uint32_t pid;
    uint32_t ppid;
    std::string nombre_programa;
    EstadoProceso estado_actual;
    uint16_t PC;
    uint16_t SP;
    int AC;
    uint8_t PSW_modo;
    uint32_t direccion_base_ram;
    uint32_t limite_memoria_bytes;
    int prioridad;
    uint64_t tiempo_cpu_ms;
    uint8_t archivos_abiertos;
};

std::string estadoToString(EstadoProceso estado) {
    switch (estado) {
        case EstadoProceso::NUEVO:      return "NUEVO";
        case EstadoProceso::LISTO:      return "LISTO";
        case EstadoProceso::EJECUTANDO: return "EJECUTANDO";
        case EstadoProceso::BLOQUEADO:  return "BLOQUEADO";
        case EstadoProceso::SALIENTE:   return "SALIENTE";
        default:                        return "DESCONOCIDO";
    }
}

std::string eventoToString(EventoProceso evento) {
    switch (evento) {
        case EventoProceso::ADMITIR:      return "Admisión del proceso por el SO";
        case EventoProceso::DESPACHAR:    return "Asignación de CPU por el planificador";
        case EventoProceso::FIN_QUANTUM:  return "Expiración de quantum (interrupción de reloj)";
        case EventoProceso::SOLICITAR_ES: return "Solicitud de operación de E/S";
        case EventoProceso::FIN_ES:       return "Interrupción por fin de E/S";
        case EventoProceso::TERMINAR:     return "Finalización del programa (exit)";
        default:                          return "Evento desconocido";
    }
}

bool simularTransicion(BCP& proceso, EventoProceso evento) {
    EstadoProceso previo = proceso.estado_actual;
    EstadoProceso siguiente = previo;
    bool cambio_valido = false;

    switch (evento) {
        case EventoProceso::ADMITIR:
            if (previo == EstadoProceso::NUEVO) {
                siguiente = EstadoProceso::LISTO;
                cambio_valido = true;
            }
            break;
        case EventoProceso::DESPACHAR:
            if (previo == EstadoProceso::LISTO) {
                siguiente = EstadoProceso::EJECUTANDO;
                cambio_valido = true;
            }
            break;
        case EventoProceso::FIN_QUANTUM:
            if (previo == EstadoProceso::EJECUTANDO) {
                proceso.tiempo_cpu_ms += 10;
                siguiente = EstadoProceso::LISTO;
                cambio_valido = true;
            }
            break;
        case EventoProceso::SOLICITAR_ES:
            if (previo == EstadoProceso::EJECUTANDO) {
                siguiente = EstadoProceso::BLOQUEADO;
                cambio_valido = true;
            }
            break;
        case EventoProceso::FIN_ES:
            if (previo == EstadoProceso::BLOQUEADO) {
                siguiente = EstadoProceso::LISTO;
                cambio_valido = true;
            }
            break;
        case EventoProceso::TERMINAR:
            if (previo == EstadoProceso::EJECUTANDO) {
                siguiente = EstadoProceso::SALIENTE;
                cambio_valido = true;
            }
            break;
    }

    if (cambio_valido) {
        proceso.estado_actual = siguiente;
        std::cout << "[TRANSICIÓN] PID " << proceso.pid << " (" << proceso.nombre_programa
                  << ") pasó de " << estadoToString(previo) << " a " << estadoToString(siguiente)
                  << " | Motivo: " << eventoToString(evento) << ".\n";
    } else {
        std::cout << "[SIN CAMBIO] PID " << proceso.pid << " (" << proceso.nombre_programa
                  << ") permanece en " << estadoToString(previo)
                  << " porque '" << eventoToString(evento) << "' no aplica en este estado.\n";
    }

    return cambio_valido;
}

int main() {
    BCP proc{102, 101, "g++_compiler", EstadoProceso::BLOQUEADO, 0x08A2, 0x0E00, 0, 0,
             0x20000, 131072, 10, 340, 4};

    simularTransicion(proc, EventoProceso::FIN_ES);      // BLOQUEADO -> LISTO
    simularTransicion(proc, EventoProceso::DESPACHAR);   // LISTO -> EJECUTANDO
    simularTransicion(proc, EventoProceso::FIN_QUANTUM); // EJECUTANDO -> LISTO (+10 ms CPU)

    return 0;
}

Writing bcp_transiciones.cpp


In [6]:
!g++ -O3 -std=c++17 bcp_transiciones.cpp -o bcp_transiciones
!./bcp_transiciones

[TRANSICIÓN] PID 102 (g++_compiler) pasó de BLOQUEADO a LISTO | Motivo: Interrupción por fin de E/S.
[TRANSICIÓN] PID 102 (g++_compiler) pasó de LISTO a EJECUTANDO | Motivo: Asignación de CPU por el planificador.
[TRANSICIÓN] PID 102 (g++_compiler) pasó de EJECUTANDO a LISTO | Motivo: Expiración de quantum (interrupción de reloj).


## Gestión Avanzada y Ciclo de Vida de Procesos.

### 1.-Análisis de Procesos Zombie en Sistemas Linux Reales Contexto: En los sistemas operativos tipo Unix, un proceso zombie ocurre cuando un hijo termina su ejecución, pero su padre aún no recolecta su estado de salida mediante la llamada wait().


**Visualización de un Proceso Zombie en Consola**

Para identificar y visualizar un proceso en estado **zombie** en la interfaz de comandos de Linux, se utilizan principalmente los utilitarios `ps` y `top`:

1. **Comando** **ps aux** **(o** **ps -ef** **)**:
  * En la columna **STAT** (Estado del proceso), un proceso zombie se identifica explícitamente con la letra **Z**.
  * En la columna de comandos (**COMMAND**), el kernel y el utilitario suelen etiquetar al ejecutable finalizado agregando la marca **

2. **Comando** **top** **(o** **htop** **)**:
* En el panel informativo de la cabecera superior, la sección de resumen de tareas incluye un contador global explícito de procesos zombie en tiempo real (por ejemplo: `Tasks: 180 total, 1 running, 178 sleeping, 0 stopped, 1 zombie`).
* En la lista de tareas inferior, la columna **S** (Estado) reflejará el carácter **Z** para el proceso en cuestión

**Análisis: ¿Por qué el Kernel Mantiene el BCP (`task_struct`) en Memoria?**

Cuando un proceso finaliza su ejecución voluntaria o involuntariamente invocando la llamada al sistema `exit()`, el kernel de Linux libera de inmediato su espacio de memoria principal (segmentos de código, datos y pila) y retira sus recursos asignados, como los descriptores de archivos abiertos. Sin embargo, el sistema operativo retiene intencionalmente la estructura de datos del procesador **task\_struct** (el Bloque de Control de Proceso o BCP) en la tabla de procesos en estado **zombie (** **Z** **)** porque el proceso padre requiere consultar el resultado de dicha ejecución. Dado que la palabra de estado de terminación (*exit status*), el código de retorno y las estadísticas contables del proceso hijo no pueden almacenarse en el BCP del padre —quien podría tener múltiples hijos ejecutándose de forma concurrente—, la única forma segura de preservar esta información es mantener la entrada mínima del BCP en memoria hasta que el padre ejecute la llamada al sistema `wait()` o `waitpid()`. Solo cuando el padre ejecuta dicha lectura, el kernel completa la recolección del estado, elimina definitivamente el BCP de la tabla de procesos y libera su identificador numérico (PID) para ser reutilizado.

### 2.- Automatización de la Adopción de Huérfanos Contexto: En el código base actual, la adopción de procesos huérfanos requiere invocar de manera explícita el método atender_huerfanos(). En un sistema operativo real, el kernel gestiona esto de forma automática cuando un proceso padre muere.

In [7]:
%%writefile desafio2_huerfanos.cpp

#include <iostream>
#include <vector>
#include <string>
#include <iomanip>
#include <cstdint>

enum class Estado { NUEVO, LISTO, EJECUTANDO, BLOQUEADO, ZOMBIE, TERMINADO };

std::string estadoToString(Estado e) {
    switch (e) {
        case Estado::NUEVO:      return "NUEVO";
        case Estado::LISTO:      return "LISTO";
        case Estado::EJECUTANDO: return "EJECUTANDO";
        case Estado::BLOQUEADO:  return "BLOQUEADO";
        case Estado::ZOMBIE:     return "ZOMBIE";
        case Estado::TERMINADO:  return "TERMINADO";
        default:                 return "DESCONOCIDO";
    }
}

struct BCP {
    int pid;
    int ppid;
    std::string nombre_programa;
    Estado estado;
    uint16_t PC;
    uint16_t SP;
    int PSW_modo;
    int codigo_salida;

    void imprimir() const {
        std::cout << "[BCP PID=" << std::setw(3) << pid
                  << " | PPID=" << std::setw(3) << ppid
                  << " | Prog: " << std::left << std::setw(12) << nombre_programa
                  << " | Estado: " << std::setw(10) << estadoToString(estado)
                  << " | PC: 0x" << std::hex << std::uppercase << std::setw(4) << std::setfill('0') << PC << std::dec << std::setfill(' ')
                  << " | Modo: " << (PSW_modo == 1 ? "KERNEL" : "USUARIO") << "]\n";
    }
};

class SistemaOperativoSimulado {
private:
    std::vector<BCP> tabla_procesos;
    int contador_pid = 100;
    int pid_activo = -1;

    // Adopción automática de hijos cuando un proceso padre muere
    void adoptar_huerfanos_internamente(int pid_padre_muerto) {
        for (auto& proc : tabla_procesos) {
            if (proc.pid != pid_padre_muerto && proc.ppid == pid_padre_muerto) {
                proc.ppid = 1;
                std::cout << "  [ADOPCIÓN AUTO] Proceso Huérfano PID " << proc.pid
                          << " (" << proc.nombre_programa << ") reasignado a init (PPID=1)\n";
            }
        }
    }

public:
    SistemaOperativoSimulado() {
        BCP init{1, 0, "init", Estado::EJECUTANDO, 0x0010, 0x0FFF, 1, 0};
        tabla_procesos.push_back(init);
        pid_activo = 1;
    }

    int simular_fork(int pid_padre, const std::string& nombre_hijo) {
        BCP* padre = obtenerBCP(pid_padre);
        if (!padre) return -1;

        contador_pid++;
        BCP hijo;
        hijo.pid = contador_pid;
        hijo.ppid = padre->pid;
        hijo.nombre_programa = nombre_hijo;
        hijo.estado = Estado::LISTO;
        hijo.PC = padre->PC;
        hijo.SP = padre->SP - 0x0100;
        hijo.PSW_modo = 0;
        hijo.codigo_salida = 0;

        tabla_procesos.push_back(hijo);
        std::cout << "  [FORK] Proceso Padre PID " << pid_padre
                  << " creó al Proceso Hijo PID " << hijo.pid << " (" << nombre_hijo << ")\n";
        return hijo.pid;
    }

    bool simular_exec(int pid, const std::string& nuevo_programa) {
        BCP* proc = obtenerBCP(pid);
        if (!proc) return false;

        std::cout << "  [EXEC] Proceso PID " << pid << " reemplaza su imagen por '" << nuevo_programa << "'\n";
        proc->nombre_programa = nuevo_programa;
        proc->PC = 0x1000;
        return true;
    }

    void simular_exit(int pid, int status) {
        BCP* proc = obtenerBCP(pid);
        if (!proc) return;

        proc->codigo_salida = status;
        proc->estado = Estado::ZOMBIE;
        std::cout << "  [EXIT] Proceso PID " << pid << " finalizó con código " << status
                  << " -> Entra a Estado ZOMBIE (esperando wait del padre)\n";

        // Detección y adopción automática de hijos activos por init (PID 1)
        adoptar_huerfanos_internamente(pid);
    }

    void simular_wait(int pid_padre) {
        for (auto it = tabla_procesos.begin(); it != tabla_procesos.end(); ++it) {
            if (it->ppid == pid_padre && it->estado == Estado::ZOMBIE) {
                std::cout << "  [WAIT] Padre PID " << pid_padre
                          << " recolectó estado del Hijo ZOMBIE PID " << it->pid
                          << ". Eliminando BCP de la tabla.\n";
                tabla_procesos.erase(it);
                return;
            }
        }
    }

    void atender_huerfanos(int pid_padre_muerto) {
        adoptar_huerfanos_internamente(pid_padre_muerto);
    }

    BCP* obtenerBCP(int pid) {
        for (auto& proc : tabla_procesos) {
            if (proc.pid == pid) return &proc;
        }
        return nullptr;
    }

    void mostrarTablaProcesos() const {
        std::cout << "\n=================================================================================\n";
        std::cout << "               ESTADO ACTUAL DE LA TABLA DE PROCESOS DEL SISTEMA                 \n";
        std::cout << "=================================================================================\n";
        for (const auto& proc : tabla_procesos) {
            proc.imprimir();
        }
        std::cout << "=================================================================================\n\n";
    }
};

int main() {
    SistemaOperativoSimulado so;

    int pid_bash   = so.simular_fork(1, "bash");
    int pid_worker = so.simular_fork(pid_bash, "worker_task");
    so.mostrarTablaProcesos();

    // Muere bash (PID 101) sin llamar a atender_huerfanos() desde main()
    so.simular_exit(pid_bash, 0);
    so.mostrarTablaProcesos();

    (void)pid_worker;
    return 0;
}

Writing desafio2_huerfanos.cpp


In [8]:
!g++ -O3 -std=c++17 desafio2_huerfanos.cpp -o desafio2_huerfanos
!./desafio2_huerfanos

  [FORK] Proceso Padre PID 1 creó al Proceso Hijo PID 101 (bash)
  [FORK] Proceso Padre PID 101 creó al Proceso Hijo PID 102 (worker_task)

               ESTADO ACTUAL DE LA TABLA DE PROCESOS DEL SISTEMA                 
[BCP PID=  1 | PPID=  0 | Prog: init         | Estado: EJECUTANDO | PC: 0x1000 | Modo: KERNEL]
[BCP PID=101 | PPID=1   | Prog: bash         | Estado: LISTO      | PC: 0x1000 | Modo: USUARIO]
[BCP PID=102 | PPID=101 | Prog: worker_task  | Estado: LISTO      | PC: 0x1000 | Modo: USUARIO]

  [EXIT] Proceso PID 101 finalizó con código 0 -> Entra a Estado ZOMBIE (esperando wait del padre)
  [ADOPCIÓN AUTO] Proceso Huérfano PID 102 (worker_task) reasignado a init (PPID=1)

               ESTADO ACTUAL DE LA TABLA DE PROCESOS DEL SISTEMA                 
[BCP PID=1   | PPID=0   | Prog: init         | Estado: EJECUTANDO | PC: 0x1000 | Modo: KERNEL]
[BCP PID=101 | PPID=1   | Prog: bash         | Estado: ZOMBIE     | PC: 0x1000 | Modo: USUARIO]
[BCP PID=102 | PPID=1   | Prog: w

### 3.- Extensión del Modelo de Estados (Estado Suspendido) Contexto: Los modelos de procesos reales suelen incorporar más estados para manejar suspensiones temporales por intervención del usuario o del sistema (por ejemplo, mediante señales como SIGSTOP y SIGCONT).

In [9]:
%%writefile desafio3_suspendido.cpp

#include <iostream>
#include <vector>
#include <string>
#include <iomanip>
#include <cstdint>

enum class Estado { NUEVO, LISTO, EJECUTANDO, BLOQUEADO, SUSPENDIDO, ZOMBIE, TERMINADO };

std::string estadoToString(Estado e) {
    switch (e) {
        case Estado::NUEVO:      return "NUEVO";
        case Estado::LISTO:      return "LISTO";
        case Estado::EJECUTANDO: return "EJECUTANDO";
        case Estado::BLOQUEADO:  return "BLOQUEADO";
        case Estado::SUSPENDIDO: return "SUSPENDIDO";
        case Estado::ZOMBIE:     return "ZOMBIE";
        case Estado::TERMINADO:  return "TERMINADO";
        default:                 return "DESCONOCIDO";
    }
}

struct BCP {
    int pid;
    int ppid;
    std::string nombre_programa;
    Estado estado;
    uint16_t PC;
    uint16_t SP;
    int PSW_modo;
    int codigo_salida;

    void imprimir() const {
        std::cout << "[BCP PID=" << std::setw(3) << pid
                  << " | PPID=" << std::setw(3) << ppid
                  << " | Prog: " << std::left << std::setw(12) << nombre_programa
                  << " | Estado: " << std::setw(10) << estadoToString(estado)
                  << " | PC: 0x" << std::hex << std::uppercase << std::setw(4) << std::setfill('0') << PC << std::dec << std::setfill(' ')
                  << " | Modo: " << (PSW_modo == 1 ? "KERNEL" : "USUARIO") << "]\n";
    }
};

class SistemaOperativoSimulado {
private:
    std::vector<BCP> tabla_procesos;
    int contador_pid = 100;

public:
    SistemaOperativoSimulado() {
        BCP init{1, 0, "init", Estado::EJECUTANDO, 0x0010, 0x0FFF, 1, 0};
        tabla_procesos.push_back(init);
    }

    int simular_fork(int pid_padre, const std::string& nombre_hijo) {
        BCP* padre = obtenerBCP(pid_padre);
        if (!padre) return -1;

        contador_pid++;
        BCP hijo{contador_pid, padre->pid, nombre_hijo, Estado::LISTO,
                 padre->PC, static_cast<uint16_t>(padre->SP - 0x0100), 0, 0};
        tabla_procesos.push_back(hijo);
        std::cout << "  [FORK] Padre PID " << pid_padre << " creó a Hijo PID "
                  << hijo.pid << " (" << nombre_hijo << ")\n";
        return hijo.pid;
    }

    // Cambia el estado del proceso a SUSPENDIDO y registra traza de contexto
    void simular_suspension(int pid) {
        BCP* proc = obtenerBCP(pid);
        if (!proc) {
            std::cout << "  [SUSPENSIÓN] Error: no existe el proceso con PID " << pid << ".\n";
            return;
        }
        if (proc->estado == Estado::ZOMBIE || proc->estado == Estado::TERMINADO) {
            std::cout << "  [SUSPENSIÓN] El proceso PID " << pid << " ya terminó ("
                      << estadoToString(proc->estado) << "); no se suspende.\n";
            return;
        }

        Estado estado_anterior = proc->estado;
        proc->estado = Estado::SUSPENDIDO;

        std::cout << "  [TRAZA SUSPENSIÓN / SIGSTOP] PID " << pid << " (" << proc->nombre_programa << "): "
                  << estadoToString(estado_anterior) << " -> SUSPENDIDO | "
                  << "Contexto guardado: PC=0x" << std::hex << std::uppercase << std::setw(4) << std::setfill('0') << proc->PC
                  << ", SP=0x" << std::setw(4) << std::setfill('0') << proc->SP << std::dec << std::setfill(' ')
                  << ", Modo=" << (proc->PSW_modo == 1 ? "KERNEL" : "USUARIO") << "\n";
    }

    BCP* obtenerBCP(int pid) {
        for (auto& proc : tabla_procesos) {
            if (proc.pid == pid) return &proc;
        }
        return nullptr;
    }

    void mostrarTablaProcesos() const {
        std::cout << "\n=================================================================================\n";
        std::cout << "               ESTADO ACTUAL DE LA TABLA DE PROCESOS DEL SISTEMA                 \n";
        std::cout << "=================================================================================\n";
        for (const auto& proc : tabla_procesos) {
            proc.imprimir();
        }
        std::cout << "=================================================================================\n\n";
    }
};

int main() {
    SistemaOperativoSimulado so;
    int pid_proc = so.simular_fork(1, "editor_texto");
    so.mostrarTablaProcesos();

    so.simular_suspension(pid_proc);
    so.mostrarTablaProcesos();

    return 0;
}

Writing desafio3_suspendido.cpp


In [10]:
!g++ -O3 -std=c++17 desafio3_suspendido.cpp -o desafio3_suspendido
!./desafio3_suspendido

  [FORK] Padre PID 1 creó a Hijo PID 101 (editor_texto)

               ESTADO ACTUAL DE LA TABLA DE PROCESOS DEL SISTEMA                 
[BCP PID=  1 | PPID=  0 | Prog: init         | Estado: EJECUTANDO | PC: 0x1000 | Modo: KERNEL]
[BCP PID=101 | PPID=1   | Prog: editor_texto | Estado: LISTO      | PC: 0x1000 | Modo: USUARIO]

  [TRAZA SUSPENSIÓN / SIGSTOP] PID 101 (editor_texto): LISTO -> SUSPENDIDO | Contexto guardado: PC=0x1000, SP=0xEFF0, Modo=USUARIO

               ESTADO ACTUAL DE LA TABLA DE PROCESOS DEL SISTEMA                 
[BCP PID=1   | PPID=0   | Prog: init         | Estado: EJECUTANDO | PC: 0x1000 | Modo: KERNEL]
[BCP PID=101 | PPID=1   | Prog: editor_texto | Estado: SUSPENDIDO | PC: 0x1000 | Modo: USUARIO]



## Concurrencia Avanzada y Robustez en IPC.

### 1.- Auditoría de Recursos IPC en Linux.
Contexto: Los recursos de IPC creados por los programas en sistemas Unix deben ser gestionados cuidadosamente por el kernel para evitar fugas de recursos (resource leaks).

In [11]:
!ipcs -m
!ipcs -s


------ Shared Memory Segments --------
key        shmid      owner      perms      bytes      nattch     status      


------ Semaphore Arrays --------
key        semid      owner      perms      nsems     



En los listados de recursos IPC de System V (ipcs -m para memoria compartida e ipcs -s para semáforos), la columna key representa la clave numérica (o 0x00000000 si el recurso es privado) utilizada para identificar o asociar el recurso entre procesos; el identificador (shmid o semid) es el número único asignado por el kernel a esa instancia en particular para referenciarla en las llamadas del sistema o liberarla; y owner indica el usuario propietario que creó el recurso, lo que permite auditar permisos y detectar recursos que quedaron activos sin liberarse (resource leaks) tras finalizar una ejecución.

### 2.- Escalabilidad a Múltiples Productores y Consumidores.
Contexto: En escenarios reales de sistemas operativos, múltiples hilos o procesos intentan acceder concurrentemente a los recursos compartidos.



In [12]:
%%writefile laboratorio_ipc_semaphoro.cpp

#include <iostream>
#include <thread>
#include <vector>
#include <chrono>
#include <string>
#include <semaphore.h>
#include <fcntl.h>
#include <unistd.h>
#include <sys/wait.h>

constexpr int TAMANO_BUFFER = 5;
constexpr int NUM_PRODUCTORES = 2;
constexpr int NUM_CONSUMIDORES = 2;
constexpr int ITEMS_POR_PRODUCTOR = 5; // 2 productores * 5 = 10 ítems en total

int buffer[TAMANO_BUFFER];
int in_idx = 0;
int out_idx = 0;

sem_t sem_huecos;    // Espacios libres disponibles en el buffer circular
sem_t sem_elementos; // Ítems listos para ser consumidos
sem_t sem_mutex;     // Exclusión mutua al modificar buffer, in_idx y out_idx

void productor(int id) {
    for (int i = 1; i <= ITEMS_POR_PRODUCTOR; ++i) {
        int dato = id * 100 + i;

        sem_wait(&sem_huecos);
        sem_wait(&sem_mutex);

        // --- Región crítica ---
        buffer[in_idx] = dato;
        std::cout << "  [PRODUCTOR " << id << "] Produjo e insertó item " << dato
                  << " en posición " << in_idx << "\n";
        in_idx = (in_idx + 1) % TAMANO_BUFFER;
        // ----------------------

        sem_post(&sem_mutex);
        sem_post(&sem_elementos);

        std::this_thread::sleep_for(std::chrono::milliseconds(120));
    }
}

void consumidor(int id, int items_a_consumir) {
    for (int i = 0; i < items_a_consumir; ++i) {
        sem_wait(&sem_elementos);
        sem_wait(&sem_mutex);

        // --- Región crítica ---
        int dato = buffer[out_idx];
        std::cout << "    [CONSUMIDOR " << id << "] Extrajo y consumió item " << dato
                  << " de posición " << out_idx << "\n";
        out_idx = (out_idx + 1) % TAMANO_BUFFER;
        // ----------------------

        sem_post(&sem_mutex);
        sem_post(&sem_huecos);

        std::this_thread::sleep_for(std::chrono::milliseconds(180));
    }
}

void probar_tubería_pipe() {
    std::cout << "\n=================================================================\n";
    std::cout << "   DEMOSTRACIÓN DE COMUNICACIÓN POR TUBERÍA (PIPE EN LINUX)      \n";
    std::cout << "=================================================================\n";

    int fd[2]; // fd[0] = lectura, fd[1] = escritura
    if (pipe(fd) == -1) {
        std::cerr << "Error al crear la tubería\n";
        return;
    }

    pid_t pid = fork();
    if (pid < 0) {
        std::cerr << "Error en fork()\n";
        close(fd[0]);
        close(fd[1]);
        return;
    }

    if (pid == 0) {
        close(fd[0]); // Cierra lectura en el hijo
        std::string mensaje = "¡Saludos desde el Proceso Hijo mediante Pipe POSIX!";
        std::cout << "  [PROCESO HIJO] Enviando mensaje por fd[1]...\n";
        ssize_t n = write(fd[1], mensaje.c_str(), mensaje.length() + 1);
        if (n == -1) {
            std::cerr << "  [PROCESO HIJO] Error al escribir en pipe\n";
        }
        close(fd[1]);
        _exit(0);
    } else {
        close(fd[1]); // Cierra escritura en el padre
        char buffer_lectura[128];
        std::cout << "  [PROCESO PADRE] Esperando lectura en fd[0]...\n";
        ssize_t leidos = read(fd[0], buffer_lectura, sizeof(buffer_lectura) - 1);
        if (leidos > 0) {
            buffer_lectura[leidos] = '\0';
            std::cout << "  [PROCESO PADRE] Mensaje recibido del Hijo: \"" << buffer_lectura << "\"\n";
        }
        close(fd[0]);
        wait(NULL);
    }
}

int main() {
    std::cout << "=================================================================\n";
    std::cout << "   PRODUCTOR-CONSUMIDOR CON 2 PRODUCTORES Y 2 CONSUMIDORES       \n";
    std::cout << "=================================================================\n\n";

    sem_init(&sem_huecos, 0, TAMANO_BUFFER);
    sem_init(&sem_elementos, 0, 0);
    sem_init(&sem_mutex, 0, 1);

    std::vector<std::thread> productores;
    std::vector<std::thread> consumidores;

    int total_items = NUM_PRODUCTORES * ITEMS_POR_PRODUCTOR; // 10 ítems
    int cuota_base = total_items / NUM_CONSUMIDORES;         // 5 por consumidor
    int resto = total_items % NUM_CONSUMIDORES;

    // Instanciar 2 hilos productores simultáneos
    for (int p = 1; p <= NUM_PRODUCTORES; ++p) {
        productores.emplace_back(productor, p);
    }

    // Instanciar 2 hilos consumidores simultáneos
    for (int c = 1; c <= NUM_CONSUMIDORES; ++c) {
        int cuota = cuota_base + (c <= resto ? 1 : 0);
        consumidores.emplace_back(consumidor, c, cuota);
    }

    for (auto& t : productores) t.join();
    for (auto& t : consumidores) t.join();

    sem_destroy(&sem_huecos);
    sem_destroy(&sem_elementos);
    sem_destroy(&sem_mutex);

    probar_tubería_pipe();

    return 0;
}

Writing laboratorio_ipc_semaphoro.cpp


In [13]:
!g++ -O3 -std=c++17 laboratorio_ipc_semaphoro.cpp -o laboratorio_ipc_semaphoro -pthread
!./laboratorio_ipc_semaphoro

   PRODUCTOR-CONSUMIDOR CON 2 PRODUCTORES Y 2 CONSUMIDORES       

  [PRODUCTOR 1] Produjo e insertó item 101 en posición 0
    [CONSUMIDOR 1] Extrajo y consumió item 101 de posición 0
  [PRODUCTOR 2] Produjo e insertó item 201 en posición 1
    [CONSUMIDOR 2] Extrajo y consumió item 201 de posición 1
  [PRODUCTOR 1] Produjo e insertó item 102 en posición 2
  [PRODUCTOR 2] Produjo e insertó item 202 en posición 3
    [CONSUMIDOR 2] Extrajo y consumió item 102 de posición 2
    [CONSUMIDOR 1] Extrajo y consumió item 202 de posición 3
  [PRODUCTOR 1] Produjo e insertó item 103 en posición 4
  [PRODUCTOR 2] Produjo e insertó item 203 en posición 0
    [CONSUMIDOR 1] Extrajo y consumió item 103 de posición 4
    [CONSUMIDOR 2] Extrajo y consumió item 203 de posición 0
  [PRODUCTOR 1] Produjo e insertó item 104 en posición 1
  [PRODUCTOR 2] Produjo e insertó item 204 en posición 2
  [PRODUCTOR 1] Produjo e insertó item 105 en posición 3
  [PRODUCTOR 2] Produjo e insertó item 205 en posición

### 3.- Depuración y Análisis de Descriptores en Tuberías (Pipe).
Contexto: La correcta gestión de los descriptores de archivos (file descriptors) es vital en las tuberías para evitar bloqueos permanentes (deadlocks) por lecturas colgadas en la espera de un fin de archivo (EOF).

**Análisis para tu celda de texto**

Por qué se requiere int fd[2]: La llamada pipe(fd) crea un canal unidireccional en el kernel y devuelve dos descriptores de archivo independientes: fd[0] (extremo de lectura) y fd[1] (extremo de escritura). Tener ambos separados permite que cada proceso cierre el extremo que no utiliza tras fork(), evitando interferencias, fugas de descriptores o que un mismo proceso lea sus propios datos por error.

Qué ocurre si el padre olvida cerrar su extremo de escritura (fd[1]) antes de read(fd[0], ...): El kernel solo envía End-Of-File (EOF, retorno 0 en read()) cuando todos los descriptores de escritura asociados a la tubería han sido cerrados. Si el padre mantiene abierto su fd[1], el kernel considera que aún existe un escritor activo aunque el hijo ya haya terminado y cerrado el suyo; como resultado, read(fd[0], ...) en el padre puede quedarse colgado/bloqueado indefinidamente (deadlock por espera de EOF) una vez consumidos los bytes existentes.

# Formato de Entrega / Portafolio de Evidencias
1. Guarda este cuaderno con todas las salidas ejecutadas incluyendo las respuesta a las preguntas (`Archivo` -> `Guardar una copia en Drive`).
2. Sube los cambios a tu repositorio de GitHub de la materia bajo el nombre `JUAN PEREZ Practica_Sistemas Operativos U2.ipynb` usando la integración directa (`Archivo` -> `Guardar una copia en GitHub`).
3. Adjunta en tu entregable de plataforma Moodle, el enlace de GitHub.